# What do the scaled profiles actually look like?

The fits are settled next door (`mtanh_fit_quality/`). This notebook takes them,
applies the campaign's four axes through `DischargePhysics.apply_mtanh_full`, and
looks at every profile that would be handed to CHEASE-BS — at the box edges and
at intermediate steps — to answer one question: **are these reasonable plasmas?**

Both notebooks import `pedestal_scan.py` at the repo root. It owns the discharge
list, the fit settings, the pedestal rule, the axes and the literature bounds, so
the numbers here and the verdict there cannot drift apart.

**Four axes, electron channel only.** $T_{e,\text{ped}}$, $n_{e,\text{ped}}$,
$\Delta T_e$, $\Delta n_e$. The KBM drive is $\nabla p_e$, which those four set
between them, and Boyle 2011 quotes exactly these quantities. The ion channel is
not unscanned: scaling $n_e$ rewrites $n_i$ and $n_z$ through quasineutrality
inside `DischargePhysics`, and $p_e$ is read off the transformed object rather
than reconstructed by hand.

**±30% ceiling.** Nothing moves a profile further. Two independent reasons that
agree: the survey spec's reference is Hatch's ±10–30% grid around the
experimental pre-ELM state, and nothing beyond ~1.4 has ever been through
cheaseBS.

**What "matching Boyle" means here.** His bands are ranges across a *population*
of lithium-scan discharges, not a target for one discharge to span. An axis is
doing its job when its reachable interval **overlaps** the band; asking one
plasma to sweep the whole band inside ±30% is not the goal and mostly is not
possible.

In [ ]:
import sys, json, pathlib
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "pedestal_scan.py").exists())
sys.path.insert(0, str(ROOT))

from pedestal_scan import (Campaign, AXES, SCALE_SANITY, BOYLE_WIDTHS, TARGETS,
                           CORE_FRAC, CORE_RHO, COL, ANALYSIS_RADII)

camp = Campaign()
shots = camp.shots

# How many steps to walk across each axis's emitted box, edges included. This is
# the "intermediate scalings" the profiles are checked at -- an odd number keeps
# a mid-box point in the set.
N_STEPS = 5

print(f"axes: {', '.join(AXES)}")
print(f"scale ceiling: {SCALE_SANITY}   steps per axis: {N_STEPS}")
print(f"targets: {TARGETS}")
print("\nregime per discharge (from its own nominal widths):")
for s in shots:
    w = camp.widths[s]
    pinned = [v for v in ("Te", "ne") if camp[s].var[v].get("pinned")]
    print(f"  {s}: {camp.regime[s]:<9} ("
          + ", ".join(f"{k} {v:.1f}%" for k, v in w.items()) + ")"
          + (f"   width unreliable: {'/'.join(pinned)} b_pos on bound"
             if pinned else ""))

## First: are the fits still the ones we approved?

Scaling multiplies fit parameters, so a fit that changed underneath this
notebook changes the meaning of every scale factor in it. This is the same check
`mtanh_fit_quality/` makes, reproduced here in two views so nothing downstream
is read on trust: **full radius** (what `apply_mtanh_full` writes and CHEASE-BS
reshapes on) and **pedestal zoom** (where the axes act).

The number on each panel is the rms inside that discharge's verdict window, as
a fraction of profile range; green is under the 1% trust threshold. If anything
here disagrees with the fit notebook, `pedestal_scan.FIT_KWARGS` moved and the
bounds below are stale.

In [ ]:
from pedestal_scan import TRUST, window_stats

FIT_VARS = ("Te", "ne", "pe")


def fit_check(zoom):
    """Data vs fit for the variables the axes act on. zoom=True: pedestal only."""
    fig, ax = plt.subplots(len(FIT_VARS), len(shots),
                           figsize=(3.4 * len(shots), 2.3 * len(FIT_VARS)),
                           squeeze=False)
    for j, shot in enumerate(shots):
        d = camp[shot]
        xl = (d.win[0] - 0.05, 1.0) if zoom else (0.0, 1.0)
        m = (d.x >= xl[0]) & (d.x <= xl[1])
        for i, var in enumerate(FIT_VARS):
            a, f = ax[i][j], d.var[var]
            a.plot(d.x[m], f["y"][m], ".", ms=3, color="0.55", label="data")
            a.plot(d.x[m], f["yhat"][m], "-", lw=1.5, color=COL[var], label="fit")
            r = window_stats(d.x, f["err"], d.win)[0]
            a.text(0.03, 0.08, f"win {r*100:.2f}%  |  all "
                               f"{f['rms_global']*100:.2f}%", fontsize=7,
                   transform=a.transAxes,
                   color="green" if r <= TRUST else "tab:red")
            a.axvspan(*d.win, color="tab:green", alpha=0.10)
            for r0 in ANALYSIS_RADII[shot]:
                a.axvline(r0, color="k", lw=0.8, ls=":", alpha=0.8)
            a.set_xlim(*xl); a.tick_params(labelsize=7)
            if i == 0:
                a.set_title(f"{shot}  {camp.regime[shot]}", fontsize=10)
            if j == 0:
                a.set_ylabel(var)
            if i == len(FIT_VARS) - 1:
                a.set_xlabel("rho_tor")
    ax[0][-1].legend(fontsize=6)
    fig.suptitle(("pedestal zoom — shaded: verdict window, dotted: GENE radii"
                  if zoom else
                  "full radius — the profile apply_mtanh_full writes"))
    plt.tight_layout()
    return fig


fit_check(zoom=False); plt.show()
fit_check(zoom=True);  plt.show()

## The bounds: what each axis reaches inside ±30%

`nominal` is where the discharge already sits. `reach` is the metric interval at
the two ceiling edges, **measured** rather than extrapolated. `covers` is the
fraction of the Boyle/target band inside that interval — read it as overlap, not
as a score.

The chart is the primary read. Each panel is the measured metric against scale
factor with the band shaded, so two things become visible that the table cannot
show: whether the response is **linear** (the box edges are computed by
inverting a one-probe linear model — 132588 `ne_width` visibly leaves the line,
and that axis is the one whose fit has `b_pos` pinned on its bound), and **how
far the band sits** in the axis's own units.

In [ ]:
rows = camp.axis_table()

hdr = (f"{'shot':>7} {'axis':<16} {'nominal':>9} {'target':>11} "
       f"{f'reach at {SCALE_SANITY}':>21} {'unit':<6} {'covers':>7}  nominal")
print(hdr); print("-" * len(hdr))
for r in rows:
    lo, hi = r["target"]
    inside = lo <= r["nominal"] <= hi
    print(f"{r['shot']:>7} {r['axis']:<16} {r['nominal']:>9.3f} "
          f"{f'{lo}-{hi}':>11} {r['reach'][0]:>9.3f}-{r['reach'][1]:<11.3f} "
          f"{r['unit']:<6} {r['cover']:>6.0%}  "
          f"{'in band' if inside else 'OUTSIDE band'}"
          + ("   [fit pinned]" if r["pinned"] else ""))
print(f"\ncovers = fraction of the band reachable within {SCALE_SANITY}. A band "
      "is a population of discharges; overlap is what matters, not spanning it.")

PROBE = np.linspace(SCALE_SANITY[0], SCALE_SANITY[1], 5)
fig, axr = plt.subplots(len(AXES), len(shots),
                        figsize=(3.2 * len(shots), 2.1 * len(AXES)),
                        squeeze=False)
for i, axis in enumerate(AXES):
    disp = AXES[axis][4]
    for j, shot in enumerate(shots):
        ax = axr[i][j]
        d = camp[shot]
        r = next(q for q in rows if q["shot"] == shot and q["axis"] == axis)
        lo, hi = r["target"]
        meas = [d.metric(axis, s) * disp for s in PROBE]
        ax.axhspan(lo, hi, color="tab:green", alpha=0.15, label="Boyle/target")
        ax.plot(PROBE, r["nominal"] * (1 + r["slope"] * (PROBE - 1)), "-",
                color="0.6", lw=1.0, label="linear model")
        ax.plot(PROBE, meas, "o", ms=4, color=COL.get(AXES[axis][0], "k"),
                label="measured")
        ax.plot([1.0], [r["nominal"]], "*", ms=10, color="k", label="nominal")
        ax.tick_params(labelsize=7)
        if i == 0:
            ax.set_title(str(shot), fontsize=10)
        if j == 0:
            ax.set_ylabel(f"{axis}\n[{r['unit']}]", fontsize=7)
        if i == len(AXES) - 1:
            ax.set_xlabel("scale factor")
axr[0][-1].legend(fontsize=5)
fig.suptitle("axis response inside the +/-30% ceiling — measured points against "
             "the linear model the box edges are computed from")
plt.tight_layout(); plt.show()

## The box, and the intermediate points that will be run

Emitted per discharge: each axis clipped to ±30%, then walked back toward
nominal until the profile it produces is one Boyle could have observed —
positive, monotonic outward through the pedestal, and with a $p_e$ width in
band. An axis is **dropped** when its band lies further away than ±30% allows;
that is a statement about the discharge, not a failure of the pipeline.

`sg_bounds.json` is what `ScanStudy` consumes. The chart shows grey for what the
target asks and colour for what survives, so a cut-down axis and a dropped one
are distinguishable at a glance.

In [ ]:
BOX, NOTES = camp.emit_box()

for shot in shots:
    print(f"{shot}: {json.dumps(BOX[shot])}")
    for n in NOTES[shot]:
        print(f"    ! {n}")

# The scan points this box implies, per axis: edges plus intermediates.
POINTS = {shot: {axis: np.linspace(lo, hi, N_STEPS)
                 for axis, (lo, hi) in BOX[shot].items()} for shot in shots}
n_pts = sum(len(v) for p in POINTS.values() for v in p.values())
print(f"\n{n_pts} single-axis points across {sum(len(p) for p in POINTS.values())} "
      f"live axes ({N_STEPS} per axis, edges included), plus {len(shots)} nominals")

with open("sg_bounds.json", "w") as fh:
    json.dump({"form": "full", "axes": list(AXES),
               "scale_ceiling": list(SCALE_SANITY), "steps": N_STEPS,
               "regime": {str(k): v for k, v in camp.regime.items()},
               "targets": TARGETS, "boyle_widths": BOYLE_WIDTHS,
               "bounds": {str(k): v for k, v in BOX.items()},
               "points": {str(s): {a: [float(v) for v in vals]
                                   for a, vals in p.items()}
                          for s, p in POINTS.items()}}, fh, indent=1)
print("written: sg_bounds.json")

fig, axb = plt.subplots(1, len(shots), figsize=(3.4 * len(shots), 3.0),
                        sharex=True, squeeze=False)
names = list(AXES)
for j, shot in enumerate(shots):
    ax = axb[0][j]
    for k, axis in enumerate(names):
        y = len(names) - 1 - k
        r = next(q for q in rows if q["shot"] == shot and q["axis"] == axis)
        if all(np.isfinite(r["scale"])):
            ax.plot(np.clip(r["scale"], 0.0, 3.0), [y + 0.16] * 2, "-", lw=5,
                    color="0.82", solid_capstyle="butt")
        if axis in BOX[shot]:
            ax.plot(BOX[shot][axis], [y - 0.08] * 2, "-", lw=6,
                    color=COL.get(AXES[axis][0], "0.3"), solid_capstyle="butt")
            ax.plot(POINTS[shot][axis], [y - 0.08] * len(POINTS[shot][axis]),
                    "|", ms=10, color="k")
        else:
            ax.text(1.0, y - 0.08, "dropped", fontsize=6, ha="center",
                    va="center", color="tab:red")
    ax.axvline(1.0, color="k", lw=0.8)
    ax.axvspan(*SCALE_SANITY, color="tab:blue", alpha=0.05)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels([a.replace("_scale", "") for a in names[::-1]], fontsize=7)
    ax.set_xlim(SCALE_SANITY[0] - 0.15, SCALE_SANITY[1] + 0.15)
    ax.set_xlabel("scale factor")
    ax.set_title(f"{shot}  {camp.regime[shot]}", fontsize=9)
    ax.tick_params(labelsize=7)
fig.suptitle("scan box — grey: asked for by the target, colour: emitted, "
             "ticks: the points that get run")
plt.tight_layout(); plt.show()

## The profiles themselves, full radius

One figure per discharge. **A row per live axis**, six columns — $T_e$, $n_e$,
derived $p_e$, each at full radius and again zoomed on the pedestal — and one
curve per scan point coloured from the low edge (blue) to the high edge (red).
Nominal is the heavy black line.

**Why the figures have different numbers of rows.** The rows are that
discharge's *live* axes, and they differ because an axis is dropped when its
Boyle band lies further away than ±30% allows. 132543 has all four; 129015 and
129038 have three; 132588 has one (`ne_ped`). Nothing is being scanned in
secret — an absent row is an axis listed as dropped in the box above.

**Why some panels show only the black line.** Each row scans *one* axis, so the
columns for the other species do not move: on a `Te_ped_scale` row the $n_e$
panel is a single curve, and on an `ne_ped_scale` row the $T_e$ panel is. The
$p_e$ column always moves, since $p_e = n_e T_e$. The one asymmetry that is not
cosmetic: scaling $n_e$ also rewrites $n_i$ and $n_z$ underneath through
quasineutrality, while scaling $T_e$ leaves $T_i$ alone — which is why the ion
channel needs no axis of its own.

This is the check that matters before submitting, because it is literally the
object CHEASE-BS is handed. Read it for:

- **separation** — curves bunched on nominal mean an axis that buys nothing;
- **ordering** — the colours should progress monotonically, no crossings;
- **the core** — the axes are named after the pedestal, so a large core change
  is the transform leaking (Stefanikova's core Gaussian is anchored to
  `a_height` at $r=0$ only, so scaling `b_height`/`b_width` drags the core);
- **the ion channel** — the $n_e$ column also moves $n_i$/$n_z$ underneath,
  which is why $p_e$ is read off the transformed object.

In [ ]:
GAL_VARS = ("Te", "ne", "pe")
VIEWS = (("full", None), ("ped", "zoom"))     # full radius, then pedestal zoom


def prof(q, var):
    """pe derived, as CHEASE builds it; everything else straight off the object."""
    if var == "pe":
        return (np.asarray(q.ds["ne"].values, dtype=float)
                * np.asarray(q.ds["Te"].values, dtype=float))
    return np.asarray(q.ds[var].values, dtype=float)


SWEEP = {}          # (shot, axis, s) -> transformed DischargePhysics, reused below
cmap = plt.get_cmap("coolwarm")

for shot in shots:
    d = camp[shot]
    live = list(POINTS[shot])
    if not live:
        print(f"{shot}: no live axes — nothing to plot")
        continue
    ncol = len(GAL_VARS) * len(VIEWS)
    fig, axg = plt.subplots(len(live), ncol,
                            figsize=(2.7 * ncol, 2.3 * len(live)), squeeze=False)
    for i, axis in enumerate(live):
        scales = POINTS[shot][axis]
        scaled = [SWEEP.setdefault((shot, axis, float(sc)), d.scaled(axis, sc))
                  for sc in scales]
        for c, (view, zoom) in enumerate(VIEWS):
            xl = (d.win[0] - 0.05, 1.0) if zoom else (0.0, 1.0)
            for v, var in enumerate(GAL_VARS):
                ax = axg[i][c * len(GAL_VARS) + v]
                m = (d.x >= xl[0]) & (d.x <= xl[1])
                ax.plot(d.x[m], prof(d.phys, var)[m], "-", lw=2.4, color="k",
                        label="nominal", zorder=4)
                for k, (sc, q) in enumerate(zip(scales, scaled)):
                    ax.plot(d.x[m], prof(q, var)[m], "-", lw=1.1,
                            color=cmap(k / max(len(scales) - 1, 1)),
                            label=f"{sc:.2f}", alpha=0.95)
                ax.axvspan(*d.win, color="tab:green", alpha=0.08)
                for r0 in ANALYSIS_RADII[shot]:
                    ax.axvline(r0, color="k", lw=0.8, ls=":", alpha=0.7)
                ax.set_xlim(*xl); ax.tick_params(labelsize=6)
                # a row scans one axis, so the other species is flat here --
                # labelled rather than left to look like a plotting bug
                if var != "pe" and var != AXES[axis][0]:
                    ax.text(0.5, 0.92, "unchanged by this axis", fontsize=5.5,
                            ha="center", va="top", color="0.45",
                            transform=ax.transAxes)
                if i == 0:
                    ax.set_title(f"{var}  ({view})", fontsize=9)
                if c == 0 and v == 0:
                    ax.set_ylabel(axis.replace("_scale", "") + "\n" + var,
                                  fontsize=8)
                else:
                    ax.set_ylabel(var, fontsize=7)
                if i == len(live) - 1:
                    ax.set_xlabel("rho_tor", fontsize=8)
                if i == 0 and c == 0 and v == len(GAL_VARS) - 1:
                    ax.legend(fontsize=5, ncol=2)
    fig.suptitle(f"{shot} ({camp.regime[shot]}) — {len(live)} live axis/axes, "
                 "scan points from blue (low edge) to red (high edge)")
    plt.tight_layout(); plt.show()

## Are they reasonable? — the same points, measured

Three tests over every scan point, not just the corners.

- **$p_e$ width against Boyle's $\Delta p_e$ band** (his panel 7g). $p_e$ has no
  axis of its own — it is derived, and CHEASE builds pressure from the profiles
  regardless — which makes the band a free consistency check: individually legal
  $n_e$ and $T_e$ widths can still combine into a pedestal he never observed.
- **Core drift against budget.** Drift is the largest fractional change inside
  $\rho_t<0.5$; budget is 30% of the change the axis produced at its own pedestal
  top. Above the diagonal the knob moved the core more than the pedestal it is
  named after. **Reported, not enforced** — this is a property of scaling a
  Stefanikova fit, and enforcing it at any defensible threshold empties every
  $T_e$ box.
- **Positive and monotonic** through the pedestal, which is the minimum bar for
  submitting an equilibrium at all.

In [ ]:
checks = []
for (shot, axis, s), q in SWEEP.items():
    d = camp[shot]
    var = AXES[axis][0]
    y = np.asarray(q.ds[var].values, dtype=float)
    m = (d.x >= 0.6) & (d.x <= 1.0)
    drift, budget = d.core_drift(axis, s)
    lo_pe, hi_pe = BOYLE_WIDTHS[camp.regime[shot]]["dpe"]
    w = d.width_psin(q, "pe")
    checks.append({"shot": shot, "axis": axis, "s": s, "pe_width": w,
                   "pe_ok": lo_pe <= w <= hi_pe, "core": drift,
                   "budget": budget, "positive": float(np.min(y)) > 0,
                   "monotonic": not np.any(np.diff(y[m])
                                           > 0.02 * float(np.max(y[m])))})

bad_pe = [c for c in checks if not c["pe_ok"]]
bad_phys = [c for c in checks if not (c["positive"] and c["monotonic"])]
over = [c for c in checks if c["core"] > c["budget"]]
print(f"{len(checks)} scan points checked")
print(f"  pe width outside Boyle's band : {len(bad_pe)}")
print(f"  non-positive or non-monotonic : {len(bad_phys)}")
print(f"  core drift over budget        : {len(over)}  (reported, not enforced)")
for c in bad_phys:
    print(f"    ! {c['shot']} {c['axis']} {c['s']:.2f}: "
          f"positive={c['positive']} monotonic={c['monotonic']}")
worst = sorted(checks, key=lambda c: -c["core"])[:5]
print("\nlargest core drift:")
for c in worst:
    print(f"  {c['shot']} {c['axis']:<16} s={c['s']:.2f}  core {c['core']:>5.0%} "
          f"of budget {c['budget']:>5.0%}")

mk = {s: m for s, m in zip(shots, ("o", "s", "^", "D"))}
fig, (axw, axc) = plt.subplots(1, 2, figsize=(13, 4.4))

for c in checks:
    lo_pe, hi_pe = BOYLE_WIDTHS[camp.regime[c["shot"]]]["dpe"]
    axw.plot([lo_pe, hi_pe], [c["s"]] * 2, "-", lw=0)          # keeps autoscale sane
for shot in shots:
    lo_pe, hi_pe = BOYLE_WIDTHS[camp.regime[shot]]["dpe"]
    axw.axvspan(lo_pe, hi_pe, color="tab:green", alpha=0.10)
for c in checks:
    axw.plot(c["pe_width"], c["s"], mk[c["shot"]], ms=6,
             color=COL.get(AXES[c["axis"]][0], "k"),
             mfc="none" if not c["pe_ok"] else None)
axw.set_xlabel("pe pedestal width  [%psiN]"); axw.set_ylabel("scale factor")
axw.set_title("every scan point's pe width against the Boyle bands\n"
              "(open marker = outside its discharge's band)", fontsize=9)
axw.grid(alpha=0.3)
axw.legend(handles=[plt.Line2D([], [], ls="", marker=mk[s], color="k",
                               label=str(s)) for s in shots], fontsize=6)

for c in checks:
    axc.plot(100 * c["budget"], 100 * c["core"], mk[c["shot"]], ms=6,
             color="tab:red" if c["core"] > c["budget"] else "tab:green")
lim = [0, 1.05 * 100 * max(max(c["core"] for c in checks),
                           max(c["budget"] for c in checks))]
axc.plot(lim, lim, "k--", lw=1.0)
axc.set_xlim(*lim); axc.set_ylim(*lim); axc.grid(alpha=0.3)
axc.set_xlabel("core-drift budget  [%]"); axc.set_ylabel("core drift  [%]")
axc.set_title(f"core drift vs {CORE_FRAC:.0%} of the pedestal-top change\n"
              f"(core = largest fractional change inside rho<{CORE_RHO})",
              fontsize=9)
axc.legend(handles=[plt.Line2D([], [], ls="", marker=mk[s], color="k",
                               label=str(s)) for s in shots], fontsize=6)
plt.tight_layout(); plt.show()

## Handover

`sg_bounds.json` holds the per-discharge box, the scan points, the regime each
discharge was scored against, and the bounds they came from. Next step is
cheaseBS on these points — nominal plus the edges first, since those are the
equilibria most likely to strain the reconstruction, and that run also retires
the open reshape-limit question the ±30% ceiling is currently standing in for.

Known and deliberate, so it is not rediscovered downstream:

- **Dropped axes are dropped against Boyle, not against physics.** An axis
  disappears when its band lies further than ±30% away. If the campaign would
  rather scan ±30% regardless of band membership, that is a different
  (defensible) choice — take `SCALE_SANITY` as the box directly and skip the
  band intersection.
- **132588 contributes one axis.** Its `ne` fit sits on the `b_pos` bound, so
  the width it reports (35.7 %$\psi_N$) is not trustworthy and its response is
  visibly nonlinear. Fixing that means lowering `ped_threshold` for that
  discharge — which was tried globally and reverted for distorting the core, but
  has not been tried for one discharge with the full-radius check in hand.
- **Core drift is measured and tolerated.** The height axes move the core; the
  numbers are on the chart above. Enforcing a limit is one flag
  (`CORE_ENFORCE`-style) and costs every $T_e$ axis.